[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_11b_Human_in_the_Loop.ipynb)

# Lesson 11b — Human-in-the-Loop (HITL) with LangGraph
**Bonus Deep-Dive | Gourav's AI Engineering Curriculum**

---

## Why this lesson exists

Lesson 11 covered LangGraph's core mechanics: StateGraph, nodes, edges, conditional branching, and MemorySaver checkpointing. But there's one critical production pattern we only touched on: **Human-in-the-Loop (HITL)**.

HITL is what separates a *demo agent* from a *trustworthy production agent*. In the real world:
- You don't want your agent to **delete files** without asking.
- You don't want it to **send emails** on your behalf without review.
- You don't want it to **charge a credit card** autonomously.
- You want a human to **correct bad reasoning** before it compounds.

HITL is the mechanism that makes AI agents safe to deploy in high-stakes situations.

---

## What you will build

| Section | What You Learn | Code Pattern |
|---|---|---|
| 1 | The HITL concept + LangGraph's model | Mental model |
| 2 | `interrupt_before` — pause before a node | Compile-time interrupt |
| 3 | `interrupt()` inside a node — dynamic pause | Runtime interrupt |
| 4 | `Command(resume=)` — resume with human input | Resuming a paused graph |
| 5 | Injecting state with `graph.update_state()` | State surgery |
| 6 | **Capstone** — Dangerous-Ops Agent with approval gate | Full HITL pattern |

---

> **Prereq:** You should have done Lesson 11 (LangGraph). The concepts of StateGraph, nodes, edges, MemorySaver, and thread_id are assumed knowledge here.

## Setup

In [ ]:
# Install dependencies
!pip install -qU langgraph langchain-anthropic anthropic

In [ ]:
import os
from google.colab import userdata  # Remove this line if running locally

# Set your Anthropic API key
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# If running locally, instead do:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

print("Environment ready ✅")

---
## Section 1 — The Mental Model: How HITL Works in LangGraph

### The key insight: checkpointing = time travel

Remember MemorySaver from Lesson 11? It saves the graph's entire state after every node. That's not just for memory across turns — it's the engine that makes HITL possible.

Here's what happens during a HITL interrupt:

```
graph.invoke(input, config)
      │
      ▼
   [node_a]  ──runs──►  state saved to checkpoint
      │
      ▼
   [node_b]  ──INTERRUPT!──►  graph STOPS here
                               returns GraphInterrupt exception
                               state is FROZEN in checkpoint
      │
      │  ◄── human looks at state, decides what to do
      │
graph.invoke(Command(resume=decision), config)
      │
      ▼
   [node_b]  ──resumes──►  continues with human's input baked in
      │
      ▼
   [node_c]  ──runs──►  done
```

### Three ways to interrupt

| Method | When | Use case |
|---|---|---|
| `interrupt_before=["node_name"]` in `compile()` | Always, before that node | Approval before every execution of a node |
| `interrupt_after=["node_name"]` in `compile()` | Always, after that node | Review node output before continuing |
| `interrupt(value)` inside a node | Conditionally, at runtime | Dynamic HITL — only interrupt when needed |

### The `Command` object — resuming a paused graph

When you resume a paused graph, you pass a `Command(resume=<value>)` object. The `value` is whatever the human decided — `"approve"`, `"reject"`, a corrected string, a dict of new state, anything.

The `interrupt()` call inside the node **returns** this value when the graph resumes. So you can write code like:

```python
def my_node(state):
    # This line PAUSES the graph and waits for human input
    human_decision = interrupt({"question": "Do you approve?", "data": state})
    
    # This line runs AFTER the human responds
    if human_decision == "approve":
        return do_the_thing(state)
    else:
        return {"status": "rejected"}
```

Elegant. The interrupt is invisible to the node — it just looks like a function call that returns the human's answer.

---
## Section 2 — `interrupt_before`: The Compile-Time Approval Gate

The simplest form. You tell LangGraph at compile time: "always pause before running this node."

Good for: nodes that perform irreversible actions (send email, delete file, charge card).

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
import json

# --- State ---
class EmailState(TypedDict):
    draft: str         # The email draft
    recipient: str     # Who it's going to
    status: str        # "drafted", "approved", "sent", "rejected"

# --- Nodes ---
def draft_email(state: EmailState) -> EmailState:
    """Node 1: Draft the email (always runs)"""
    draft = f"""Subject: Meeting Tomorrow

Hi {state['recipient']},

Just a reminder that we have a meeting tomorrow at 10am.
Please come prepared with your weekly updates.

Best,
Gourav"""
    print(f"📝 Email drafted for {state['recipient']}")
    return {"draft": draft, "status": "drafted"}

def send_email(state: EmailState) -> EmailState:
    """Node 2: Send the email (will be interrupted before this runs)"""
    # In real life: call Gmail API, SMTP, etc.
    print(f"📨 Email SENT to {state['recipient']}")
    print(f"Content:\n{state['draft']}")
    return {"status": "sent"}

# --- Build the graph ---
builder = StateGraph(EmailState)
builder.add_node("draft_email", draft_email)
builder.add_node("send_email", send_email)

builder.set_entry_point("draft_email")
builder.add_edge("draft_email", "send_email")
builder.add_edge("send_email", END)

# KEY LINE: interrupt_before tells LangGraph to PAUSE before send_email
checkpointer = MemorySaver()
graph = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["send_email"]   # <-- The magic
)

print("Graph compiled with interrupt_before=['send_email'] ✅")

In [ ]:
# Run the graph — it will stop BEFORE send_email
config = {"configurable": {"thread_id": "email-thread-1"}}

initial_state = {
    "recipient": "team@company.com",
    "draft": "",
    "status": ""
}

print("=" * 50)
print("RUNNING GRAPH (will pause before send_email)")
print("=" * 50)

# invoke() returns the state at the point of interruption
result = graph.invoke(initial_state, config)

print("\n" + "=" * 50)
print("GRAPH PAUSED! Current state:")
print("=" * 50)
print(f"Status: {result['status']}")
print(f"Draft:\n{result['draft']}")
print("\n⏸️  Waiting for human approval...")

In [ ]:
# --- HUMAN DECISION POINT ---
# In a real app, this would be a web form, Slack message, CLI prompt, etc.
# Here we simulate it:

human_says = input("Do you approve sending this email? (yes/no): ")

if human_says.lower() == "yes":
    print("\n✅ Approved! Resuming graph...")
    # Resume: pass Command(resume=None) to just continue
    # For interrupt_before, resuming means "go ahead and run the node"
    final = graph.invoke(Command(resume=None), config)
    print(f"\nFinal status: {final['status']}")
else:
    print("\n❌ Rejected. Email will NOT be sent.")
    # Just don't resume — or update state to mark rejection
    graph.update_state(config, {"status": "rejected"})
    state = graph.get_state(config)
    print(f"Final status: {state.values['status']}")

### What just happened?

1. Graph ran `draft_email` → saved checkpoint → hit the interrupt gate before `send_email` → **stopped**.
2. You (the human) inspected the draft and decided.
3. You either resumed (graph ran `send_email`) or rejected (state was updated manually).

**Key lesson:** The graph's state is frozen in the checkpoint between steps 1 and 3. You can wait minutes, hours, or days. The MemorySaver holds the state until you resume. In production you'd use a database-backed checkpointer (PostgreSQL, Redis) instead of the in-memory one.

---
## Section 3 — `interrupt()` Inside a Node: Dynamic HITL

`interrupt_before` is all-or-nothing — it always pauses. But what if you only want to interrupt sometimes? For example: only ask for approval if the action is HIGH RISK.

The `interrupt()` function called inside a node gives you full control.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt

# --- State ---
class OperationState(TypedDict):
    operation: str      # What to do: "read", "write", "delete"
    target: str         # What file/resource
    result: str         # Outcome
    approved: bool      # Was it approved?

# Risk levels — delete is dangerous, write is medium, read is fine
RISK_LEVELS = {
    "read": "LOW",
    "write": "MEDIUM",
    "delete": "HIGH"
}

# --- Nodes ---
def assess_and_execute(state: OperationState) -> OperationState:
    """Single node that decides WHETHER to interrupt based on risk"""
    op = state["operation"]
    target = state["target"]
    risk = RISK_LEVELS.get(op, "UNKNOWN")
    
    print(f"\n🔍 Operation: {op.upper()} on '{target}'")
    print(f"⚠️  Risk level: {risk}")
    
    # Dynamic decision: only interrupt for HIGH or MEDIUM risk
    if risk in ("HIGH", "MEDIUM"):
        print(f"🛑 Risk is {risk} — requesting human approval...")
        
        # interrupt() pauses the graph and sends data to the human
        # It RETURNS the human's response when the graph resumes
        human_response = interrupt({
            "message": f"Approve {op} on '{target}'?",
            "risk": risk,
            "operation": op,
            "target": target
        })
        
        # Code below this line runs AFTER the human responds
        if human_response != "approve":
            print(f"❌ Human rejected the operation")
            return {"result": "Operation rejected by human", "approved": False}
        
        print(f"✅ Human approved!")
    
    # Execute the operation (simulated)
    if op == "read":
        result = f"Read contents of '{target}': [file contents here]"
    elif op == "write":
        result = f"Wrote data to '{target}' successfully"
    elif op == "delete":
        result = f"DELETED '{target}' permanently"
    else:
        result = "Unknown operation"
    
    print(f"⚡ Executed: {result}")
    return {"result": result, "approved": True}

# --- Build graph ---
builder = StateGraph(OperationState)
builder.add_node("assess_and_execute", assess_and_execute)
builder.set_entry_point("assess_and_execute")
builder.add_edge("assess_and_execute", END)

checkpointer = MemorySaver()
# NOTE: No interrupt_before here! The interrupt is handled INSIDE the node dynamically
graph2 = builder.compile(checkpointer=checkpointer)

print("Dynamic HITL graph ready ✅")

In [ ]:
# Test 1: LOW RISK — should run without interruption
print("=" * 50)
print("TEST 1: Low-risk READ operation")
print("=" * 50)

config1 = {"configurable": {"thread_id": "op-thread-read"}}
result1 = graph2.invoke(
    {"operation": "read", "target": "report.pdf", "result": "", "approved": False},
    config1
)
print(f"\n📋 Result: {result1['result']}")
print("ℹ️  No interruption — read is LOW risk")

In [ ]:
# Test 2: HIGH RISK — should interrupt and wait
print("=" * 50)
print("TEST 2: High-risk DELETE operation")
print("=" * 50)

config2 = {"configurable": {"thread_id": "op-thread-delete"}}

# This will pause at the interrupt() call inside the node
result2 = graph2.invoke(
    {"operation": "delete", "target": "production_database.db", "result": "", "approved": False},
    config2
)

print("\n⏸️  Graph paused — waiting for human input")
print(f"Interrupt data sent to human: {result2}")

In [ ]:
# Now simulate the human responding
# In production: this comes from a webhook, Slack bot, web form, etc.

human_decision = input("\nType 'approve' to allow deletion, anything else to reject: ")

print(f"\n🧑 Human decided: '{human_decision}'")
print("▶️  Resuming graph with human's decision...")

# Command(resume=value) — this is how human input gets INTO the graph
# The interrupt() call in the node will RETURN this value
final2 = graph2.invoke(Command(resume=human_decision), config2)

print(f"\n📋 Final result: {final2['result']}")
print(f"✅ Approved: {final2['approved']}")

### The power of `interrupt()` vs `interrupt_before`

| | `interrupt_before` | `interrupt()` |
|---|---|---|
| **When set** | At compile time | At runtime inside a node |
| **Condition** | Always interrupts | You control when |
| **Human data** | Just the current state | Custom dict you define |
| **Human return value** | Only `None` (just continue) | Any value you want |
| **Best for** | Always-require-approval nodes | Conditional / risk-based approval |

In production, you'll use `interrupt()` far more often because most real systems need *conditional* human involvement.

---
## Section 4 — `graph.update_state()`: Human Edits the State Directly

Sometimes you don't want to just approve/reject — you want to **correct** what the agent produced. Like: "Your draft email is good but change the subject line."

`graph.update_state()` lets you surgically modify the checkpoint before resuming.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt

# --- State ---
class ReportState(TypedDict):
    topic: str
    draft: str
    human_feedback: str
    final_report: str
    iterations: int

# --- Nodes ---
def write_draft(state: ReportState) -> ReportState:
    """Write an initial draft"""
    topic = state["topic"]
    iteration = state.get("iterations", 0) + 1
    
    feedback = state.get("human_feedback", "")
    
    if feedback:
        draft = f"""[Revision {iteration}] Report on: {topic}

INCORPORATING FEEDBACK: {feedback}

Executive Summary: This report covers the key aspects of {topic}.
The analysis shows promising results with room for improvement.
Recommendations have been updated based on reviewer feedback.

[Revised based on: {feedback}]"""
    else:
        draft = f"""[Draft {iteration}] Report on: {topic}

Executive Summary: This report covers the key aspects of {topic}.
The analysis shows promising results.
See appendix for detailed methodology."""
    
    print(f"\n📝 Draft {iteration} written")
    return {"draft": draft, "iterations": iteration}

def review_gate(state: ReportState) -> ReportState:
    """Ask human to review — either approve or provide feedback"""
    print(f"\n--- DRAFT FOR REVIEW ---")
    print(state["draft"])
    print("--- END DRAFT ---")
    
    # interrupt() pauses here
    # We pass the draft so the human can see it
    response = interrupt({
        "action": "review",
        "draft": state["draft"],
        "instruction": "Reply 'approve' to finalize, or give feedback to revise"
    })
    
    return {"human_feedback": response}

def should_revise(state: ReportState) -> str:
    """Conditional edge: go back to write or move to finalize"""
    if state["human_feedback"].lower() == "approve":
        return "finalize"
    return "revise"

def finalize_report(state: ReportState) -> ReportState:
    """Polish the draft into final report"""
    final = f"""=== FINAL APPROVED REPORT ===

{state['draft']}

=== END OF REPORT ===
Approved after {state['iterations']} iteration(s)."""
    print("\n✅ Report finalized!")
    return {"final_report": final}

# --- Build graph with revision loop ---
builder = StateGraph(ReportState)
builder.add_node("write_draft", write_draft)
builder.add_node("review_gate", review_gate)
builder.add_node("finalize_report", finalize_report)

builder.set_entry_point("write_draft")
builder.add_edge("write_draft", "review_gate")
builder.add_conditional_edges(
    "review_gate",
    should_revise,
    {"revise": "write_draft", "finalize": "finalize_report"}
)
builder.add_edge("finalize_report", END)

checkpointer3 = MemorySaver()
graph3 = builder.compile(checkpointer=checkpointer3)

print("Human-in-the-loop revision graph ready ✅")
print("Flow: write → review (human) → [revise → write → review]* → finalize")

In [ ]:
# Run the revision loop
config3 = {"configurable": {"thread_id": "report-thread-1"}}

print("=" * 50)
print("STARTING REPORT GENERATION")
print("=" * 50)

state = graph3.invoke(
    {"topic": "AI Agent Safety", "draft": "", "human_feedback": "", "final_report": "", "iterations": 0},
    config3
)

print("\n⏸️  Waiting for your review...")

# Multi-turn revision loop
while True:
    feedback = input("\nYour review (type 'approve' to finalize, or give feedback): ")
    
    print(f"\n▶️  Resuming with: '{feedback}'")
    state = graph3.invoke(Command(resume=feedback), config3)
    
    # Check if graph finished (final_report is set)
    if state.get("final_report"):
        print("\n" + "=" * 50)
        print("FINAL REPORT:")
        print("=" * 50)
        print(state["final_report"])
        break
    # Otherwise graph paused again for another review
    print("\n⏸️  Draft revised — waiting for your review again...")

### What you just built

A **human-in-the-loop revision loop**:

```
write_draft → review_gate ──(approve)──► finalize_report → END
                    ▲                           
                    │◄──(feedback)──────────────┘
                    │
                 write_draft (revision)
```

This pattern is used in real production AI systems:
- **Coding assistants**: Agent writes code → human reviews → request changes or approve.
- **Content pipelines**: Agent writes article → editor reviews → revise or publish.
- **Data pipelines**: Agent generates SQL → analyst checks → run or reject.

---
## Section 5 — `graph.update_state()`: Surgery on the Checkpoint

Sometimes you want to inspect and directly modify the state while the graph is paused — without resuming through the normal flow. Think of it as a debugger for your agent.

In [ ]:
# Using graph2 from Section 3 (the delete operation graph)
# Let's demonstrate state inspection and modification

from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt

config_demo = {"configurable": {"thread_id": "state-surgery-demo"}}

# Kick off a high-risk operation (will pause)
result = graph2.invoke(
    {"operation": "write", "target": "config.yaml", "result": "", "approved": False},
    config_demo
)

print("Graph paused. Let's inspect the checkpoint...")

In [ ]:
# --- Inspect the paused state ---
snapshot = graph2.get_state(config_demo)

print("📸 CHECKPOINT SNAPSHOT:")
print(f"  Values: {snapshot.values}")
print(f"  Next nodes to run: {snapshot.next}")
print(f"  Tasks pending: {snapshot.tasks}")

In [ ]:
# --- Modify state BEFORE resuming ---
# Imagine: the human wants to redirect the write to a different file

print("🔧 Human modifying state: changing target file...")

# update_state PATCHES the state in the checkpoint
# as_node tells LangGraph which node is 'making' this update
graph2.update_state(
    config_demo,
    {"target": "config_backup.yaml"},  # Override the target
    as_node="assess_and_execute"       # Pretend this node made the change
)

# Verify the change
snapshot_after = graph2.get_state(config_demo)
print(f"✅ Target updated to: {snapshot_after.values['target']}")

# Now resume — the node will use the updated target
print("\n▶️  Resuming with updated state...")
final = graph2.invoke(Command(resume="approve"), config_demo)
print(f"\n📋 Final result: {final['result']}")

### `get_state` + `update_state` = full debugger

These two tools give you complete visibility and control over a paused agent:

- `graph.get_state(config)` → read the frozen state, see what node runs next
- `graph.update_state(config, patch)` → rewrite any field in the state before resuming
- `graph.get_state_history(config)` → see every checkpoint ever saved (time travel!)

This is incredibly powerful for debugging production agents — you can replay, rewind, and fix state without restarting the whole run.

---
## Section 6 — Capstone: The Dangerous-Ops Agent

Now let's put it all together into a realistic agent that:
1. Uses an LLM to decide what operations to perform
2. Has a **risk classifier** that flags dangerous operations
3. **Interrupts** for human approval on risky ops
4. Only executes after getting the green light

This is the pattern used in real agentic systems like coding agents, DevOps bots, and data pipeline orchestrators.

In [ ]:
import anthropic
import json
from typing import TypedDict, Literal, Optional, List, Any
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt

client = anthropic.Anthropic()

# ============================================================
# TOOL DEFINITIONS (what the LLM can call)
# ============================================================
TOOLS = [
    {
        "name": "read_file",
        "description": "Read the contents of a file. Safe operation.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "File path to read"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file. Overwrites existing content.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"}
            },
            "required": ["path", "content"]
        }
    },
    {
        "name": "delete_file",
        "description": "Permanently delete a file. IRREVERSIBLE.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "list_directory",
        "description": "List files in a directory. Safe operation.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"}
            },
            "required": ["path"]
        }
    }
]

# Risk classification
TOOL_RISK = {
    "read_file": "LOW",
    "list_directory": "LOW",
    "write_file": "MEDIUM",
    "delete_file": "HIGH"
}

print("Tools defined ✅")

In [ ]:
# ============================================================
# STATE
# ============================================================
class AgentState(TypedDict):
    user_request: str
    messages: List[dict]           # Full conversation history
    pending_tool: Optional[dict]   # Tool call waiting for approval
    tool_results: List[dict]       # Results of executed tools
    final_answer: str              # Agent's final response
    status: str                    # "running", "awaiting_approval", "done", "rejected"

print("State defined ✅")

In [ ]:
# ============================================================
# SIMULATED TOOL EXECUTOR (in real life: actual file system calls)
# ============================================================
def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Simulate tool execution — replace with real implementations"""
    if tool_name == "read_file":
        return f"[Contents of {tool_input['path']}]: Hello world! This is sample file content."
    elif tool_name == "list_directory":
        return f"[Files in {tool_input['path']}]: file1.txt, config.yaml, data.csv, secret.db"
    elif tool_name == "write_file":
        return f"[SUCCESS] Wrote {len(tool_input['content'])} bytes to {tool_input['path']}"
    elif tool_name == "delete_file":
        return f"[SUCCESS] Permanently deleted {tool_input['path']}"
    return "[ERROR] Unknown tool"

# ============================================================
# NODES
# ============================================================
def llm_node(state: AgentState) -> AgentState:
    """Node 1: Ask LLM what to do next"""
    print("\n🤖 LLM thinking...")
    
    # Build messages — if we have tool results, feed them back
    messages = state.get("messages", [])
    
    if not messages:
        # First turn
        messages = [{"role": "user", "content": state["user_request"]}]
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        system="""You are a file system agent. When given a task, use the available tools to complete it.
        Always prefer reading/listing before writing or deleting.
        When you have completed the task, respond with your final answer (no tool call).""",
        tools=TOOLS,
        messages=messages
    )
    
    # Check if LLM wants to use a tool
    if response.stop_reason == "tool_use":
        tool_block = next(b for b in response.content if b.type == "tool_use")
        tool_name = tool_block.name
        tool_input = tool_block.input
        risk = TOOL_RISK.get(tool_name, "UNKNOWN")
        
        print(f"🔧 LLM wants to call: {tool_name}({tool_input}) [Risk: {risk}]")
        
        # Add assistant's response to messages (including the tool_use block)
        updated_messages = messages + [{"role": "assistant", "content": response.content}]
        
        return {
            "messages": updated_messages,
            "pending_tool": {
                "id": tool_block.id,
                "name": tool_name,
                "input": tool_input,
                "risk": risk
            },
            "status": "has_tool_call"
        }
    else:
        # LLM gave a final answer
        final_text = next((b.text for b in response.content if hasattr(b, "text")), "Done.")
        print(f"✅ LLM finished: {final_text}")
        return {
            "messages": messages + [{"role": "assistant", "content": response.content}],
            "final_answer": final_text,
            "status": "done"
        }

def approval_gate(state: AgentState) -> AgentState:
    """Node 2: Approve or reject based on risk"""
    tool = state["pending_tool"]
    risk = tool["risk"]
    
    if risk == "LOW":
        # Auto-approve low-risk operations
        print(f"   ✅ Auto-approved (LOW risk)")
        return {"status": "approved"}
    else:
        # HIGH or MEDIUM: ask human
        print(f"\n🚨 HUMAN APPROVAL REQUIRED")
        print(f"   Tool: {tool['name']}")
        print(f"   Input: {tool['input']}")
        print(f"   Risk: {risk}")
        
        decision = interrupt({
            "message": f"Approve {tool['name']} on {tool['input']}?",
            "tool": tool,
            "instructions": "Reply 'approve' or 'reject'"
        })
        
        if decision == "approve":
            print("   ✅ Human approved")
            return {"status": "approved"}
        else:
            print("   ❌ Human rejected")
            return {"status": "rejected"}

def execute_node(state: AgentState) -> AgentState:
    """Node 3: Execute the approved tool"""
    tool = state["pending_tool"]
    print(f"\n⚡ Executing: {tool['name']}({tool['input']})")
    
    result = execute_tool(tool["name"], tool["input"])
    print(f"   Result: {result}")
    
    # Feed result back to LLM in next turn
    tool_result_message = {
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool["id"],
            "content": result
        }]
    }
    
    return {
        "messages": state["messages"] + [tool_result_message],
        "pending_tool": None,
        "tool_results": state.get("tool_results", []) + [{"tool": tool["name"], "result": result}],
        "status": "running"
    }

def rejected_node(state: AgentState) -> AgentState:
    """Node 4: Handle rejection — tell LLM the tool was rejected"""
    tool = state["pending_tool"]
    print(f"\n🚫 Informing LLM that {tool['name']} was rejected")
    
    rejection_message = {
        "role": "user",
        "content": [{
            "type": "tool_result",
            "tool_use_id": tool["id"],
            "content": f"[REJECTED by human] {tool['name']} was not permitted.",
            "is_error": True
        }]
    }
    
    return {
        "messages": state["messages"] + [rejection_message],
        "pending_tool": None,
        "status": "running"  # Let LLM try something else
    }

print("Nodes defined ✅")

In [ ]:
# ============================================================
# ROUTING LOGIC
# ============================================================
def route_after_llm(state: AgentState) -> str:
    if state["status"] == "done":
        return "done"
    return "has_tool_call"  # go to approval gate

def route_after_approval(state: AgentState) -> str:
    if state["status"] == "approved":
        return "approved"
    return "rejected"

# ============================================================
# BUILD THE GRAPH
# ============================================================
builder = StateGraph(AgentState)

builder.add_node("llm", llm_node)
builder.add_node("approval_gate", approval_gate)
builder.add_node("execute", execute_node)
builder.add_node("rejected", rejected_node)

builder.set_entry_point("llm")

builder.add_conditional_edges(
    "llm",
    route_after_llm,
    {"done": END, "has_tool_call": "approval_gate"}
)

builder.add_conditional_edges(
    "approval_gate",
    route_after_approval,
    {"approved": "execute", "rejected": "rejected"}
)

# Both execute and rejected loop back to LLM
builder.add_edge("execute", "llm")
builder.add_edge("rejected", "llm")

capstone_checkpointer = MemorySaver()
capstone_graph = builder.compile(checkpointer=capstone_checkpointer)

print("=" * 50)
print("CAPSTONE GRAPH COMPILED ✅")
print("=" * 50)
print("""
Flow:
  llm → (done) → END
  llm → (tool call) → approval_gate
                          │
                    (LOW risk) auto-approve
                    (MEDIUM/HIGH) interrupt → human
                          │
                    (approved) → execute → llm
                    (rejected) → rejected → llm
""")

In [ ]:
# ============================================================
# RUN THE CAPSTONE AGENT
# ============================================================
capstone_config = {"configurable": {"thread_id": "capstone-1"}}

# Try a request that will involve BOTH safe and risky operations
user_request = "List the files in /data, then delete the file called data.csv"

print("=" * 60)
print(f"USER REQUEST: {user_request}")
print("=" * 60)

state = capstone_graph.invoke(
    {
        "user_request": user_request,
        "messages": [],
        "pending_tool": None,
        "tool_results": [],
        "final_answer": "",
        "status": "running"
    },
    capstone_config
)

# Handle HITL loop
while not state.get("final_answer"):
    print("\n⏸️  PAUSED FOR HUMAN APPROVAL")
    decision = input("Your decision (approve/reject): ")
    print(f"▶️  Resuming with: '{decision}'")
    state = capstone_graph.invoke(Command(resume=decision), capstone_config)

print("\n" + "=" * 60)
print("AGENT FINAL ANSWER:")
print("=" * 60)
print(state["final_answer"])
print("\n📊 Tools used:")
for r in state.get("tool_results", []):
    print(f"  - {r['tool']}: {r['result']}")

---
## Summary — What You Learned

### The Core Idea
HITL in LangGraph works because **checkpointing freezes graph state** at any point. The graph can sleep for seconds, minutes, or days while a human makes a decision — then resume exactly where it left off.

### The Three Patterns

| Pattern | Code | When to use |
|---|---|---|
| **Always pause before node** | `compile(interrupt_before=["node"])` | Critical nodes that always need approval |
| **Conditionally pause inside node** | `interrupt(data)` → returns human input | Risk-based, dynamic approval |
| **Edit frozen state** | `graph.update_state(config, patch)` | Human correction before resume |

### The Resume Pattern
```python
# Pause
result = graph.invoke(initial_state, config)  # stops at interrupt

# Human decides...

# Resume
result = graph.invoke(Command(resume=human_value), config)  # continues
```

### Production Considerations
- **Swap MemorySaver for a real DB** — `langgraph-checkpoint-postgres` or `langgraph-checkpoint-redis` for production. MemorySaver is in-memory only.
- **Interrupts are async by nature** — in real apps, the first `invoke()` call returns immediately, you store the thread_id, and resume later via a webhook/API call.
- **The LangGraph Platform** (cloud offering) handles all of this for you with built-in HITL queues, but it's good to understand the primitives first.

---
## What's Next?

You're now in **Phase 3** of the curriculum. Next lesson: **Lesson 16 — Deployment** (FastAPI + Docker + Fly.io/Railway). You'll deploy your AutoResearcher agent as a real web service that people can actually call over the internet.

---
## Exercises (Do at least 2)

1. **Slack approval**: Instead of `input()`, mock out a Slack message send + webhook receive for the approval. What would the code look like?

2. **Timeout handling**: What happens if a human never responds? Add logic that auto-rejects after a simulated timeout (hint: check `snapshot.created_at`).

3. **Multi-approver**: Modify the capstone so HIGH risk operations require TWO humans to approve before executing. How do you store partial approvals in state?

4. **State rollback**: Use `get_state_history()` to list all checkpoints for a thread and replay from an earlier one. Try this with `graph2` after several operations.

5. **Audit log**: Add an `audit_trail: List[dict]` field to state that records every tool call, who approved it, and the timestamp. Print it at the end.